## Coles Preprocessing Script for MongoDB

Author: Anna Margarita Sofia T. Licup (Margie)

Last Modified: 09/05/2026

Note: Rebekah De Losa's MongoDBUploadScript_Woolworths.ipynb was the base for this. Modifications were made to fit the Coles scraped files.

---

### Purpose
This script takes raw Coles product pricing data (scraped as CSV files), cleans and standardizes it, matches each product to the DiscountMate product database, and uploads the pricing records to MongoDB.

**Run this script whenever new Coles scrape files are available and need to be loaded into the MongoDB database.**

### Steps
1. **Setup** - import libraries, set the data folder paths, connect to MongoDB
2. **Define helper functions** - database read/write custom functions
3. **Define target columns** - the fields required by the database schema
4. **Define processing functions** - clean and reformat each column
5. **Load & process files** - loop through all CSVs, apply processing functions, compile the results
6. **Validate** - confirm that the data is clean and complete before upload
7. **Deduplicate against MongoDB** - remove any records already in the database to avoid re-uploading
8. **Upload to MongoDB** - send the processed records to the product_pricings collection in MongoDB

### Before you run this script
- Make sure you have a `.env` file in the **same folder as this notebook** containing your MongoDB connection string.
- Update the `path` variable in **Step 1b** to point to your local folder containing the scraped CSV files.
- Update the `master_scrape_path` variable in **Step 1b** to point to the Master Coles Scrape file on your machine.
- Required Python packages: `pandas`, `numpy`, `pymongo`, `python-dotenv`

### Step 1a - Setup: Import Libraries
We import all the libraries needed before running anything else. We need to import these to make sure that we're able to use the commands from the library.

In [ ]:
# Libraries for preprocessing the data

# Imports pandas library to work with the data like a spreadsheet where there are rows and columns
import pandas as pd

# Imports the EmptyDataError to skip over any files that are empty (no to crashing!)
from pandas.errors import EmptyDataError

# Imports the numpy library to handle numbers and fill in blanks where data is missing
import numpy as np

# Imports the os library to find and open files from folders on the computer
import os

# Imports datetime and timezone to help when recording exactly when the data was uploaded
from datetime import datetime, timezone

# Imports MongoClient to connect to the MongoDB database
from pymongo import MongoClient

# Imports load_dotenv to load my database password from a hidden file (for security!)
from dotenv import load_dotenv

### Step 1b - Setup: Set Data Folder Paths

**Update both paths before running.**

- path - the folder on your machine that contains the scraped store subfolders (ex. `coles/`).
- master_scrape_path - the full path to Master_Coles_Scrape.csv, which is the old Coles master scrape file that contains the GTINs (barcodes) we need to map to new scrape files.

These create variables so we don't have to keep manually updating the path in all succeeding code blocks.

In [ ]:
# Path to the folder containing the store scrape subfolders
# Update this to your local path before running
path = r"C:\Users\margi\Capstone T1 2026\DiscountMate_new\experimental\WebAPIScraping"

# Full path to the Master Coles Scrape CSV
# This file contains the GTINs (barcodes) that are missing from the newer scrape files
# Update this to your local path before running
master_scrape_path = r"C:\Users\margi\Capstone T1 2026\DiscountMate_new\Master_Data_2026_Onward\Master_Coles_Scrape.csv"

### Step 1c - Setup: Connect to MongoDB

We load the database password from the `.env` file (so it is never written directly into this script) and use it to connect to the DiscountMate MongoDB database.

FYI: If the connection fails, check that your `.env` file exists in the same folder as this notebook and contains a valid `MONGO_URI` value.

In [ ]:
# Loads the .env file so that we can easily use the database password
load_dotenv()

# Gets the database password from the .env file and stores it in a variable for easy access
mongo_uri = os.getenv('MONGO_URI')

# Uses the password to connect to the MongoDB database, like logging in
client = MongoClient(mongo_uri)

# Decides which database we want to use for this
database = 'DiscountMate_DB'

# Opens the database
db = client[database]

### Step 2 - Define Helper Functions

These two functions handle reading from and writing to MongoDB:

1. **get_data** - retrieves all records from a given collection and returns them as a pandas DataFrame
2. **create_data** - uploads a list of processed records to a given collection one by one, printing progress so you can monitor the upload and see where it stopped if it fails midway

> **Note:** `update_data` is reserved for future use (editing existing records) and is not yet implemented.

In [ ]:
# Defines functions to use when retrieving the scraped data and uploading the files
def get_data(table, database):
    # Pulls all records from a given table in the database and turns it into a spreadsheet
    return pd.DataFrame(database[table].find({}))

# Uploads the processed data to the MongoDB database (and the progress of upload)
def create_data(db, tablename, newDocuments):
    # newDocuments: a list of processed records ready to be uploaded to the database
    # Starts a counter to keep track of how many records have been uploaded
    docCount = 0
    # Opens the target table in the database
    collection = db[tablename]
    # Goes through each record one by one
    for doc in newDocuments:
        # Uploads the current record to the database
        collection.insert_one(doc)
        # Adds 1 to the counter after each successful upload
        docCount += 1
        # Prints the current count so we can see the upload progress (out of how many are being uploaded and so we know where it fails if ever)
        print(f"{docCount}/{len(newDocuments)} uploaded - product_id: {doc['product_id']}, date: {doc['date']}")
    print(f"Upload complete. {docCount} records uploaded.")
    # Reloads the table to confirm everything was uploaded
    table = pd.DataFrame(collection.find({}))
    # Returns the updated table and the total number of records uploaded
    return table, docCount

# Placeholder for a tool for edits maybe??? (Leaving this here)
def update_data():
    return

# Load the existing products table from the database so we can match Coles products to it later
prods = get_data('products', db)

### Step 3 - Define Target Columns
This is the list of columns that the `product_pricings` collection in MongoDB expects. Every record we upload must contain these fields.

The processing functions in Step 4 are responsible for producing each of these columns from the raw Coles data.

In [ ]:
# Defines the list of columns we want to keep and upload to the database
# Checklist of info for the database
targetvars = ['product_id',      # The unique ID that identifies the product in our database
              'product_code',    # Code for the product
              'store_chain',     # The store the product was scraped from
              'date',            # The date the price was collected from the Coles website
              'price',           # The price of the product on that date
              'best_price',      # The lowest price ever seen for this product (historically)
              'best_unit_price', # The lowest unit price ever seen for this product (historically)
              'unit_price',      # The price per unit on that date
              'is_on_special',   # Whether the product was on special on that date (True or False)
              'created_at']      # The date and time this record was uploaded to the database

### Step 4 - Define Processing Functions

These functions clean and reformat the raw Coles data into the standard format used by the database:

1. **get_date** - converts the scrape timestamp into a consistent date and time format
2. **get_price** - ensures the product price is stored as a number and not text
3. **get_unit_price** - calculates the price per single unit (ex. `'0.0225 per g'`) using three fallback methods in order: (1) parse the `Comparable` column directly, (2) divide `UnitPrice` by `UnitQuantity` and `UnitMeasure`, (3) divide `Price_Now` by the product `Size`. Whichever method produces a result first is used.
4. **map_gtin_from_master** - maps the GTIN (barcode) for each product from the Master Coles Scrape file by matching on product name, since the newer scrape files do not include GTINs. This is needed so records can be matched to the `products` table in MongoDB.
5. **transform_product_id** - validates and stores `ProductId` for joining against `product_code` in MongoDB
6. **get_is_on_special** - flags whether the product was on a price-related special on that date. Only flags `True` for actual price reductions — `NEW` and `FLYBUYS` are excluded because they do not represent a price discount. To be confirmed with the DA team.
7. **join_prods** - matches each Coles record to an existing product in the database using the GTIN from the master scrape lookup

In [ ]:
def get_date(data, coltarget):
    # Converts the timestamp column into a datetime format that Python and MongoDB can work with
    data['date'] = pd.to_datetime(data[coltarget])


def get_price(data, coltarget):
    # Makes sure the price is saved as a number with decimals (not as text)
    data['price'] = data[coltarget].astype('float')


def get_unit_price(data, comparable_col, unit_price_col, size_col, unit_col, price_col, product_size_col):
    # Standardizes unit name variations so they all use the same abbreviation
    # For example, 'grams', 'gram' and 'g' all become 'g'
    size_dict = {
        'gram': 'g',
        'grams': 'g',
        'litre': 'l',
        'litres': 'l',
        'metre': 'm',
        'metres': 'm',
        'meters': 'm',
        'millilitre': 'ml',
        'millilitres': 'ml',
        'ml': 'ml',
        'kg': 'kg',
        'ea': 'each',
        'each': 'each',
        'pack': 'pack',
        'piece': 'piece',
        'pair': 'pair',
        'set': 'set',
        'cm': 'cm',
        'l': 'l',
        'g': 'g',
    }

    # --- Case 1: Parse the Comparable column directly ---
    # The Comparable column often contains a ready-made unit price string (e.g. '$2.25/100g')
    # We extract the dollar amount, quantity, and unit from it and divide to get the per-unit price
    # (e.g. '$2.25/100g' becomes '0.0225 per g')
    comparable_extracted = data[comparable_col].str.strip().str.extract(r'\$([\d\.]+)/\s*([\d\.]+)\s*(\w+)')
    comparable_price = comparable_extracted[0].astype(float)
    comparable_quantity = comparable_extracted[1].astype(float)
    comparable_unit = comparable_extracted[2].str.lower().str.strip().replace(size_dict)
    case1 = round(comparable_price / comparable_quantity, 4).astype(str) + ' per ' + comparable_unit

    # --- Case 2: Use UnitPrice, UnitQuantity, and UnitMeasure ---
    # If the Comparable column is blank, we fall back to dividing the unit price by the unit quantity
    # (e.g. $15.80 for 1 kg becomes '15.8 per kg')
    unitprice_value = pd.to_numeric(data[unit_price_col], errors='coerce')
    unit_quantity = pd.to_numeric(data[size_col], errors='coerce')
    unit_measurement = data[unit_col].str.lower().str.strip().replace(size_dict)
    case2 = round(unitprice_value / unit_quantity, 4).astype(str) + ' per ' + unit_measurement
    case2_available = unitprice_value.notna() & unit_quantity.notna() & data[unit_col].notna()

    # --- Case 3: Divide product price by product size ---
    # If both of the above are blank, we calculate unit price from the full product price and size column
    # (e.g. $6.00 for a 750g product becomes '0.008 per g')
    product_size = data[product_size_col].str.lower().str.strip()
    # Add a space between the number and unit if there isn't one (e.g. '141gram' becomes '141 gram')
    product_size = product_size.str.replace(r'(\d)([a-z])', r'\1 \2', regex=True)
    # Split the size into its quantity and unit parts
    size_split = product_size.str.extract(r'([\d\.]+)\s*(\w+)')
    size_quantity = pd.to_numeric(size_split[0], errors='coerce')
    size_unit = size_split[1].replace(size_dict)
    case3 = round(data[price_col].astype(float) / size_quantity, 4).astype(str) + ' per ' + size_unit

    # Apply the three cases in order: Case 1 first, then Case 2, then Case 3 as a last resort
    data['unit_price'] = np.where(
        comparable_extracted[0].notna(),
        case1,
        np.where(
            case2_available,
            case2,
            case3
        )
    )


def map_gtin_from_master(data, master_scrape_path):
    """
    Maps a GTIN (barcode) to each product in the current scrape file by matching on product name.

    Why this is needed:
    The newer Coles scrape files do not include a GTIN column. Without a GTIN, we cannot match
    products to the 'products' table in MongoDB. The Master Coles Scrape (an older, more complete
    scrape) does contain GTINs, so we look up each product by name and borrow its GTIN.

    Products that cannot be matched by name will have no GTIN and will be dropped in join_prods.
    """
    # Load the master scrape file, keeping gtin as text to avoid scientific notation on long barcodes
    master = pd.read_csv(master_scrape_path, dtype={'gtin': str}, low_memory=False)

    # Keep only the name-to-gtin mapping and drop any rows where either field is missing
    master = master[['name', 'gtin']].dropna(subset=['name', 'gtin'])

    # Normalize both sides to lowercase and strip whitespace for a clean match
    master['name_lower'] = master['name'].str.lower().str.strip()
    data['name_lower'] = data['Name'].str.lower().str.strip()

    # Left join: each product in the scrape gets the GTIN from the master file if a name match is found
    # Products with no name match will have NaN for gtin and will be excluded in join_prods
    data = data.merge(master[['name_lower', 'gtin']], on='name_lower', how='left')

    # Remove the temporary lowercase name column used for matching
    data.drop(columns=['name_lower'], inplace=True)

    # Replace 'nan' or '0' strings with None since they are not real barcodes
    data['gtin'] = data['gtin'].replace({'nan': None, '0': None})

    print(f"GTIN mapped: {data['gtin'].notna().sum()} of {len(data)} products matched by name")
    print(f"GTIN not found (will be dropped at join): {data['gtin'].isna().sum()}")

    return data


def transform_product_id(data, coltarget):
    # Stores the Coles ProductId as a string so it can be used to match against product_code in MongoDB
    # Removes any rows where the ProductId is missing, since we can't match these to the database
    data['product_code'] = data[coltarget].astype(str).str.strip()
    data['product_code'] = data['product_code'].replace({'nan': None, '0': None})
    data.dropna(subset=['product_code'], inplace=True)


def get_is_on_special(data, coltarget):
    # Marks a product as on special (True) only when the PromotionType column has a value AND
    # that value is not NEW, FLYBUYS, or blank.
    # NEW is excluded because it promotes a new product listing, not a price reduction.
    # FLYBUYS is excluded because it's a loyalty reward (third party), not a direct price discount.
    # Values like SPECIAL, EVERYDAY, and DOWNDOWN are treated as genuine price specials.
    # To be confirmed with the DA team.
    data['is_on_special'] = data[coltarget].notna() & ~data[coltarget].str.strip().str.upper().isin(['NEW', 'FLYBUYS', ''])


def join_prods(data, product_data):
    # Matches each Coles record to its entry in the products table using the GTIN as the common link
    # Any products that don't have a matching GTIN in the products table are dropped
    return pd.merge(product_data[['_id', 'product_code', 'gtin']].copy(),
                    data,
                    how='inner',
                    on='gtin').rename(columns={'_id': 'product_id'})

### Step 5 - Load and Process Coles Files

1. Loops through every CSV in the `coles/` subfolder, skipping non-CSV or empty files.
2. Drops duplicates caused by the same product being scraped under different `Brand_Searched` keywords — since `Brand_Searched` is the source of duplicates, it is excluded from the uniqueness check.
3. Maps a GTIN (barcode) to each product by matching its name against the Master Coles Scrape file.
4. Applies all processing functions to each file.
5. Matches each record to the `products` table using the GTIN.
6. Concatenates all processed files into a single DataFrame ready for upload.

**Column mapping - what each raw column becomes in the database:**
```
Timestamp      - date           date and time the price was collected
Price_Now      - price          current price of the product
Comparable     - unit_price     price per unit (primary source)
UnitPrice      - unit_price     price per unit (fallback if Comparable is blank)
ProductId      - product_code   used to match to product in MongoDB
PromotionType  - is_on_special  True if not NEW, FLYBUYS, or blank
gtin           - gtin           barcode mapped from Master Coles Scrape by product name
```

In [ ]:
# Creates an empty container to store each CSV file when it's loaded
df_list = {}
# The name of the store folder to look for CSV files in
store = 'coles'

# Loops through every file in the Coles folder
for file in os.listdir(os.path.join(path, store)):
    # Skips anything that isn't a CSV file
    if not file.endswith('.csv'):
        continue
    try:
        # Reads the CSV, keeping the ProductId as text to avoid scientific notation on long IDs
        df = pd.read_csv(os.path.join(path, store, file), low_memory=False, dtype={'ProductId': str})

        # Drops duplicates caused by the same product appearing under different Brand_Searched values.
        # Brand_Searched is intentionally excluded from the uniqueness check because two rows with
        # identical product details but different Brand_Searched keywords are the same product.
        df.drop_duplicates(subset=[col for col in df.columns if col != 'Brand_Searched'], keep='first', inplace=True)

        # Adds the loaded file to the container
        df_list[file] = df
        print(f"File {file} added to dataframe list...")
    except EmptyDataError:
        # Skips any CSV files that are completely empty
        print(f"File {file} is empty. Skipping...")

In [ ]:
# Creates an empty table to collect all processed records from all files before uploading
complete_df = pd.DataFrame(columns=targetvars)

# Loops through each loaded file, applies the processing functions, and adds the results to complete_df
for name, df in df_list.items():
    print(f"\nProcessing: {name}")
    print(f"Row count: {len(df)}")

    # Applies the processing functions to clean and reformat the raw data
    get_date(df, 'Timestamp')
    get_price(df, 'Price_Now')
    get_unit_price(df, 'Comparable', 'UnitPrice', 'UnitQuantity', 'UnitMeasure', 'Price_Now', 'Size')
    transform_product_id(df, 'ProductId')
    get_is_on_special(df, 'PromotionType')

    # Maps a GTIN to each product using the Master Coles Scrape file (matched by product name)
    # This step is required because the newer Coles scrape files do not include GTINs
    df = map_gtin_from_master(df, master_scrape_path)

    # Matches each record to an existing product in the products table using the GTIN
    # Any product with no GTIN match is dropped here
    joindf = join_prods(df, prods)

    # Placeholder: best_price and best_unit_price logic to be confirmed with the DA/ML team
    joindf[['best_price', 'best_unit_price']] = None

    # Labels all records as Coles so we know which store they came from
    joindf['store_chain'] = 'coles_generic'

    # Restricts to just the target columns plus gtin (gtin is kept here for deduplication in Step 7)
    joindf = joindf[targetvars[:-1] + ['gtin']].copy()

    print(f"{name} initial upload length: {len(joindf)}")

    # Removes any remaining duplicate rows based on gtin
    # This catches any cases not already resolved by the Brand_Searched drop at load time
    joindf.drop_duplicates(subset=['gtin'], inplace=True)
    print(f"{name} upload length after dropping gtin duplicates: {len(joindf)}")

    # Adds the processed file to the combined table
    complete_df = pd.concat([complete_df, joindf], ignore_index=True)

print(f"\nTotal records across all files: {len(complete_df)}")

### Step 6 - Validate Data Before Uploading

Run these checks before uploading to confirm the processed data is clean and in the correct format. If any check fails, investigate the issue before proceeding to Step 7.

1. **Target columns present** - confirms that all required columns exist in the dataframe
2. **No nulls in critical fields** - checks that `product_id`, `product_code`, `store_chain`, `date`, `price`, and `gtin` are not blank
3. **Price is valid** - checks that no prices are zero or negative
4. **Date format is correct** - confirms the date is in the expected timestamp format
5. **is_on_special is boolean** - confirms the `is_on_special` flag is stored as `True` or `False`
6. **store_chain is consistent** - confirms all records are labelled as `'coles_generic'`

If there are any issues, they will be printed so they can be investigated.

In [ ]:
# Checks if all target columns are there
# Excludes created_at as this is added at upload
upload_cols = targetvars[:-1] + ['gtin']
# Finds any columns that are missing from the dataframe
missing_cols = [c for c in upload_cols if c not in complete_df.columns]
# Prints the missing columns, or confirms all are present
print(f"Missing target columns: {missing_cols if missing_cols else 'None'}")

# Checks for null values in the fields that must not be blank
necessary = ['product_id', 'product_code', 'store_chain', 'date', 'price', 'gtin']
# Counts how many blank values exist in each required column
null_counter = complete_df[necessary].isnull().sum()
print("\nNo. of nulls in required fields:")
print(null_counter)

# Checks if price is numeric and positive (no negatives!)
# Finds any records where the price is zero or negative
invalid_prices = complete_df[complete_df['price'] <= 0]
print(f"\nRecords with price <= 0: {len(invalid_prices)}")

# Checks if date format is correct
try:
    # Tries to parse the date column as a timestamp format
    pd.to_datetime(complete_df['date'])
    # Confirms the format is correct if there is no error
    print("\nDate format check: SUCCESS")
except Exception as e:
    # Prints the error if the format is incorrect
    print(f"\nDate format check: FAILED — {e}")

# Checks that is_on_special is stored as a boolean (True or False)
print(f"\nis_on_special dtype: {complete_df['is_on_special'].dtype}")
print(f"is_on_special counter:\n{complete_df['is_on_special'].value_counts()}")

# Checks if store_chain values are all coles_generic
print(f"\nstore_chain unique values: {complete_df['store_chain'].unique()}")

# Checks the final shape of the processed file
print(f"\nFinal dataframe shape: {complete_df.shape}")
# Shows the first 5 rows of the final dataframe
complete_df[upload_cols].head()

### Step 7 - Deduplicate Against MongoDB (Skip Already-Uploaded Records)

It's common for scrape files to accumulate in the folder over time, meaning the same file may be present across multiple runs of this script. This step protects against uploading duplicate records by checking what is already in MongoDB and keeping only records that haven't been uploaded yet.

**Steps:**
1. We fetch the `product_id` and `date` of every existing Coles record from MongoDB (just these two fields and not the full records, so it's fast).
2. We then do a left join between our local processed data and the existing MongoDB records.
3. Any local record that already has a matching `product_id` + `date` combination in MongoDB is dropped. Only genuinely new records proceed to upload.

> A record is considered a duplicate if it has the same `product_id` **and** `date` as something already in the database. Two records for the same product on different dates are treated as distinct pricing snapshots and are both kept.

In [ ]:
# Fetches the product_id and date of all existing Coles records already in MongoDB.
# We only retrieve these two fields (not full records) to keep this fast.
existing_coles = pd.DataFrame(db['product_pricings'].find(
    {'store_chain': 'coles_generic'},
    {'product_id': 1, 'date': 1, '_id': 0}
))

if len(existing_coles) > 0:
    # Ensures both sides use the same datetime format before comparing
    existing_coles['date'] = pd.to_datetime(existing_coles['date'])
    complete_df['date'] = pd.to_datetime(complete_df['date'])

    # Left join — records that already exist in MongoDB will get a '_merge' value of 'both'
    merged = complete_df.merge(existing_coles, on=['product_id', 'date'], how='left', indicator=True)

    already_uploaded = merged[merged['_merge'] == 'both']
    to_upload = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

    print(f"Records already in MongoDB (will be skipped): {len(already_uploaded)}")
    print(f"New records to upload: {len(to_upload)}")

    # Replace complete_df with only the new records
    complete_df = to_upload
else:
    # No existing Coles records found — this is likely the first run, upload everything
    print("No existing Coles records found in MongoDB — uploading all records.")
    print(f"Records to upload: {len(complete_df)}")

### Step 8 - Prepare and Upload to MongoDB

**Steps:**
1. Copies the processed DataFrame and drops the `gtin` column (used for joining only as it is not stored in MongoDB)
2. Adds a `created_at` timestamp so we know when each record was uploaded
3. Replaces `NaN` values with `None` (MongoDB does not accept Python's `NaN`)
4. Converts the DataFrame to a list of dictionaries for upload
5. Uploads all new records to the `product_pricings` collection

> If the upload fails partway through, **re-run from Step 7**: `create_data` inserts one record at a time and prints progress, so you can see exactly where it stopped. Step 7 will automatically exclude anything already uploaded, so there is no risk of double-uploading.

In [ ]:
import_data = complete_df.copy()
# Remove gtin — used for joining only, not stored in MongoDB
import_data.drop(columns=['gtin'], inplace=True, errors='ignore')
# Add created_at timestamp so we know when each record was uploaded
import_data['created_at'] = pd.to_datetime('now')
# Replace NaN with None — MongoDB does not accept Python's NaN
import_data = import_data.replace({np.nan: None})
# Convert to a list of dictionaries — this is the format MongoDB expects
import_dict = import_data.to_dict(orient='records')
print(f"Import dict has {len(import_dict)} records")

In [ ]:
# Uploads all records in import_dict to the product_pricings table in MongoDB one by one
updatedPriceTable, priceCount = create_data(db, 'product_pricings', import_dict)
# Prints the total number of records successfully uploaded
print(f"Upload finished. {priceCount} product_pricings uploaded.")

---

### Known Data Issues and How They Were Handled

Updated as of May 9 2026

This section documents the known issues found in the Coles scraped data and the fixes applied in this script.

---

#### 1. Newer scrape files have no GTIN (barcode)
**Issue:**  
The newer Coles scrape files (Temp) do not include a `gtin` column. Without a GTIN, we cannot match products to the `products` table in MongoDB.

**Temp Fix:**  
The `map_gtin_from_master` function (Step 4) loads the older `Master_Coles_Scrape.csv` file, which does contain GTINs, and matches each product by name. If a name match is found, the GTIN is borrowed from the master file. Products with no name match have no GTIN and are dropped in `join_prods`.

**To investigate:**  
Whether name matching is reliable enough long-term, or whether there is a better key to join on (ex. `ProductId`). Also worth confirming whether future scrapes will include GTINs directly.

---

#### 2. `price_comparable` column is sometimes blank
**Issue:**  
The `Comparable` column, which is the primary source for unit price, is not always populated.

**Fix:**  
`get_unit_price` tries three methods in order and uses the first one that produces a result:
1. Parse `Comparable` directly (e.g. `'$2.25/100g'` → `'0.0225 per g'`)
2. Divide `UnitPrice` by `UnitQuantity` using `UnitMeasure` as the label (e.g. `$15.80 / 1 kg` to `'15.8 per kg'`)
3. Divide `Price_Now` by the product `Size` column as a last resort (e.g. `$6.00 / 750g` to `'0.008 per g'`)

If all three methods fail for a row, `unit_price` will be `NaN` for that record.

---

#### 3. Duplicate rows caused by Brand_Searched keywords
**Issue:**  
The same product can appear multiple times in a scrape file if it was returned as a result for more than one brand search keyword. All other columns (name, price, GTIN, etc.) are identical and only `Brand_Searched` differs.

**Fix:**  
At load time, `Brand_Searched` is excluded from the uniqueness check and `drop_duplicates` is applied across all other columns. This keeps one copy of each unique product record regardless of how many keyword searches returned it.

---

#### 4. Different definition of `is_on_special` for Coles
**Issue:**  
The `PromotionType` column in Coles uses values like `NEW`, `FLYBUYS`, `SPECIAL`, `EVERYDAY`, and `DOWNDOWN`. Not all of these represent a genuine price reduction.

**Temp Fix:**  
`get_is_on_special` currently marks a product as on special (`True`) for any `PromotionType` value that is not `NEW`, `FLYBUYS`, or blank. `NEW` is excluded because it marks a newly listed product, not a price cut. `FLYBUYS` is excluded because it is a third-party loyalty reward, not a direct price reduction.

**To confirm with DA team:**  
Whether `EVERYDAY` and `DOWNDOWN` should be treated as price specials or excluded.

---

#### 5. Unit pricing differences across retailers
**Issue:**  
Different retailers express unit pricing differently wherein some use per gram (`g`), others per kilogram (`kg`), per pack, or per unit. This makes cross-retailer price comparisons unreliable.

**Fix:**  
None yet in this script. Rebekah suggested that unit standardization should be handled in the `dim_product` table so that it is consistent across all retailers at the database level rather than per-script.

---

#### 6. Limited MongoDB storage
**Issue:**  
The MongoDB instance ran low on storage during earlier upload runs.

**Temp Fix:**  
Rebekah dropped 10 collections (backed up locally first) to free up space. The upload was then able to proceed. No code changes were needed.